# 🎤 Tamaki — Criador de Voz (Qwen3-TTS Voice Design)

**Antes de rodar, no menu à direita (Settings):**
- **Accelerator:** GPU T4 x2
- **Internet:** On *(precisa ter o celular verificado no Kaggle)*

Depois rode as células **de cima para baixo**, uma por vez (botão ▶).

## 1️⃣ Instalar (demora uns 2–3 minutos)

In [1]:
!pip install -q qwen-tts==0.1.1 soundfile
print("✅ Instalado!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
✅ Instalado!


In [2]:
import os, shutil, time
import numpy as np, soundfile as sf
import gradio as gr

MAX = 10
BASE = "/kaggle/working"

# Som do aviso (sininho)
_t = np.linspace(0, 0.25, int(22050 * 0.25), False)
_toque = np.concatenate([0.5*np.sin(2*np.pi*f*_t) for f in (660, 880, 1320, 880, 1320)])
SININHO = f"{BASE}/sininho.wav"
sf.write(SININHO, _toque, 22050)

def limpar_nome(nome):
    nome = (nome or "personagem").strip().lower().replace(" ", "_")
    return "".join(c for c in nome if c.isalnum() or c == "_") or "personagem"

def gerar(nome, descricao, texto, idioma, quantidade, progress=gr.Progress()):
    nome = limpar_nome(nome)
    quantidade = int(quantidade)
    pasta = f"{BASE}/{nome}_testes"
    os.makedirs(pasta, exist_ok=True)
    caminhos = []
    for i in range(1, quantidade + 1):
        progress((i - 1, quantidade), desc=f"Gerando a versão {i} de {quantidade}…")
        torch.manual_seed(i)
        wavs, sr = modelo.generate_voice_design(
            text=texto, language=idioma, instruct=descricao, do_sample=True)
        caminho = f"{pasta}/{nome}_{i:02d}.wav"
        sf.write(caminho, wavs[0], sr)
        caminhos.append(caminho)
    gr.Info("🔔 Todas as vozes estão prontas!")
    aviso = f"{BASE}/sininho_{int(time.time())}.wav"   # nome novo para tocar toda vez
    shutil.copy(SININHO, aviso)
    players = []
    for i in range(MAX):
        if i < quantidade:
            players.append(gr.update(value=caminhos[i], label=f"🎧 Versão {i+1}", visible=True))
        else:
            players.append(gr.update(value=None, visible=False))
    escolhas = gr.update(choices=[f"Versão {i}" for i in range(1, quantidade + 1)],
                         value=None, visible=True)
    return [aviso, escolhas, gr.update(visible=True)] + players

def salvar(nome, texto, escolha):
    if not escolha:
        raise gr.Error("Escolha uma versão primeiro!")
    nome = limpar_nome(nome)
    i = int(escolha.split()[-1])
    origem = f"{BASE}/{nome}_testes/{nome}_{i:02d}.wav"
    destino = f"{BASE}/{nome}_referencia.wav"
    shutil.copy(origem, destino)
    with open(f"{BASE}/{nome}_referencia.txt", "w", encoding="utf-8") as f:
        f.write(texto)
    gr.Info(f"⭐ Versão {i} salva como {nome}_referencia.wav")
    return destino, destino

with gr.Blocks(title="Criador de Voz") as tela:
    gr.Markdown("# 🎤 Criador de Voz\nDescreva a voz, gere as versões, escute e escolha a sua favorita.")
    with gr.Group():
        nome = gr.Textbox(label="Nome do personagem", value="tamaki")
        descricao = gr.Textbox(label="Descrição da voz (em inglês funciona melhor)", lines=4,
            value=("Young Brazilian Portuguese female voice, early 20s, slightly husky and raspy, "
                   "low-medium pitch, mischievous and unpredictable attitude, playful sarcasm, "
                   "confident and relaxed, a hint of a smirk in her tone, natural pacing, clean studio quality."))
        texto = gr.Textbox(label="Frase de teste", lines=2,
            value="Ah, você achou mesmo que eu ia seguir o plano? Que fofo. Relaxa, eu tenho um plano melhor… talvez.")
        with gr.Row():
            idioma = gr.Dropdown(label="Idioma", value="Portuguese",
                choices=["Portuguese", "English", "Japanese", "Spanish", "Chinese",
                         "Korean", "French", "German", "Italian", "Russian"])
            quantidade = gr.Slider(label="Quantas versões", minimum=1, maximum=MAX, step=1, value=10)
    gerar_btn = gr.Button("✨ Gerar vozes", variant="primary", size="lg")

    sininho = gr.Audio(label="🔔 Aviso", autoplay=True, visible=True, interactive=False)

    players = [gr.Audio(visible=False, interactive=False, type="filepath") for _ in range(MAX)]

    escolha = gr.Radio(label="⭐ Qual é a sua favorita?", choices=[], visible=False)
    salvar_btn = gr.Button("💾 Salvar a escolhida", variant="primary", visible=False)
    escolhida_audio = gr.Audio(label="Voz escolhida", interactive=False)
    arquivo = gr.File(label="⬇️ Baixar a voz escolhida")

    gerar_btn.click(gerar, [nome, descricao, texto, idioma, quantidade],
                    [sininho, escolha, salvar_btn] + players)
    salvar_btn.click(salvar, [nome, texto, escolha], [escolhida_audio, arquivo])

_, _, link = tela.queue().launch(share=True, inline=False, prevent_thread_lock=True)

from IPython.display import HTML, display
display(HTML(f"<div style='font-size:20px;padding:14px;background:#1565c0;border-radius:12px'>"
             f"<a href='{link}' target='_blank' style='color:white'>👉 Toque aqui para abrir o Criador de Voz</a></div>"))
print("Não compartilhe esse link: quem tiver ele usa a sua GPU do Kaggle.")

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9b799a6272d9d250d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Não compartilhe esse link: quem tiver ele usa a sua GPU do Kaggle.


## 2️⃣ Carregar o modelo (demora uns 3–5 minutos na primeira vez)

In [3]:
import torch
from qwen_tts import Qwen3TTSModel

# Placas antigas (T4/P100) não lidam bem com bfloat16, então usamos float32 nelas
cap = torch.cuda.get_device_capability(0)[0]
tipo = torch.bfloat16 if cap >= 8 else torch.float32

modelo = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=tipo,
    attn_implementation="sdpa",
)
print("✅ Modelo carregado na", torch.cuda.get_device_name(0))

/bin/sh: 1: sox: not found

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ Modelo carregado na Tesla T4


## 3️⃣ Gerar as versões da Tamaki

Pode mudar a **DESCRICAO**, o **TEXTO** e a **QUANTIDADE** aqui embaixo e rodar de novo quantas vezes quiser.

In [4]:
import os, time, shutil
import numpy as np, soundfile as sf
import ipywidgets as widgets
from IPython.display import Audio, FileLink, HTML, display

NOME = "tamaki"   # nome do personagem (sem espaços)

DESCRICAO = ("Young Brazilian Portuguese female voice, early 20s, slightly husky and raspy, "
             "low-medium pitch, mischievous and unpredictable attitude, playful sarcasm, "
             "confident and relaxed, a hint of a smirk in her tone, natural pacing, clean studio quality.")

TEXTO = ("Ah, você achou mesmo que eu ia seguir o plano? Que fofo. "
         "Relaxa, eu tenho um plano melhor… talvez.")

QUANTIDADE = 10

PASTA = f"/kaggle/working/{NOME}_testes"
os.makedirs(PASTA, exist_ok=True)

# Barra de andamento
barra = widgets.IntProgress(value=0, min=0, max=QUANTIDADE, description="Gerando:",
                            bar_style="info", layout=widgets.Layout(width="95%"))
status = widgets.HTML(f"⏳ Começando… 0 de {QUANTIDADE}")
display(barra, status)

inicio = time.time()
for i in range(1, QUANTIDADE + 1):
    status.value = f"⏳ Gerando a versão {i} de {QUANTIDADE}…"
    torch.manual_seed(i)
    wavs, sr = modelo.generate_voice_design(
        text=TEXTO, language="Portuguese", instruct=DESCRICAO, do_sample=True)
    sf.write(f"{PASTA}/{NOME}_{i:02d}.wav", wavs[0], sr)
    barra.value = i
    resta = (time.time() - inicio) / i * (QUANTIDADE - i)
    status.value = f"✅ {i} de {QUANTIDADE} prontas — falta uns {int(resta//60)} min {int(resta%60)} s"

barra.bar_style = "success"
status.value = f"🎉 As {QUANTIDADE} versões estão prontas!"

# Aviso: som + faixa verde
t = np.linspace(0, 0.25, int(22050 * 0.25), False)
toque = np.concatenate([0.5*np.sin(2*np.pi*f*t) for f in (660, 880, 1320, 880, 1320)])
display(Audio(toque, rate=22050, autoplay=True))
display(HTML("<div style='font-size:22px;padding:14px;margin:10px 0;background:#2e7d32;"
             "color:white;border-radius:12px'>🔔 Todas as vozes estão prontas! "
             "Escute e toque em <b>Escolher</b> na sua favorita.</div>"))

# Botões de escolha
def escolher(i, saida):
    def clicar(_):
        origem = f"{PASTA}/{NOME}_{i:02d}.wav"
        destino = f"/kaggle/working/{NOME}_referencia.wav"
        shutil.copy(origem, destino)
        with open(f"/kaggle/working/{NOME}_referencia.txt", "w", encoding="utf-8") as f:
            f.write(TEXTO)
        saida.clear_output()
        with saida:
            print(f"⭐ Versão {i} escolhida e salva como {NOME}_referencia.wav")
            print("⬇️ Para baixar: toque nos ⋮ do player → Baixar (ou no link azul)")
            display(Audio(destino))
            display(FileLink(f"{NOME}_referencia.wav"))
    return clicar

for i in range(1, QUANTIDADE + 1):
    display(HTML(f"<h3 style='margin:18px 0 4px'>🎧 Versão {i}</h3>"))
    display(Audio(f"{PASTA}/{NOME}_{i:02d}.wav"))
    botao = widgets.Button(description=f"Escolher a versão {i}", button_style="warning",
                           icon="star", layout=widgets.Layout(width="260px", height="44px"))
    saida = widgets.Output()
    botao.on_click(escolher(i, saida))
    display(botao, saida)

IntProgress(value=0, bar_style='info', description='Gerando:', layout=Layout(width='95%'), max=10)

HTML(value='⏳ Começando… 0 de 10')

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Button(button_style='warning', description='Escolher a versão 1', icon='star', layout=Layout(height='44px', wi…

Output()

...

Output()

Button(button_style='warning', description='Escolher a versão 9', icon='star', layout=Layout(height='44px', wi…

Output()

Button(button_style='warning', description='Escolher a versão 10', icon='star', layout=Layout(height='44px', w…

Output()

## 4️⃣ Escolher a Tamaki oficial

Troque o número em **ESCOLHIDA** pela sua favorita e rode.
O arquivo **tamaki_referencia.wav** vai aparecer em **Output** (à direita) para baixar.